In [16]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [17]:
train_df = pd.read_csv("../dataset/train_split.csv")
val_df = pd.read_csv("../dataset/val_split.csv")

class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

print("Jumlah training   :", len(train_df))
print("Jumlah validation :", len(val_df))

Jumlah training   : 21947
Jumlah validation : 5487


In [18]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32


def load_image(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(
        image,
        channels=1
    )

    image = tf.cast(
        image,
        tf.float32
    )

    # Grayscale 1 channel → RGB 3 channel
    image = tf.image.grayscale_to_rgb(image)

    # 48×48 → 64×64
    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    # Preprocessing khusus MobileNetV2
    image = preprocess_input(image)

    return image, label

In [19]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["label_index"].values
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["label_index"].values
    )
)

train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df),
    seed=42
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

val_dataset = val_dataset.batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

In [20]:
images, labels = next(iter(train_dataset))

print("Shape images :", images.shape)
print("Shape labels :", labels.shape)
print("Nilai minimum:", tf.reduce_min(images).numpy())
print("Nilai maksimum:", tf.reduce_max(images).numpy())

Shape images : (32, 64, 64, 3)
Shape labels : (32,)
Nilai minimum: -1.0
Nilai maksimum: 1.0


In [21]:
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(64, 64, 3)
)

base_model.trainable = False

inputs = Input(shape=(64, 64, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.5)(x)

outputs = Dense(7, activation="softmax")(x)

model = Model(
    inputs=inputs,
    outputs=outputs
)

model.summary()

C:\Users\BRT\AppData\Local\Temp\ipykernel_20000\2557565750.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [22]:
print("Base model trainable:", base_model.trainable)
print("Total parameters:", model.count_params())

Base model trainable: False
Total parameters: 2422855


In [23]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [24]:
print("Base model trainable:", base_model.trainable)
print("Jumlah parameter:", model.count_params())

Base model trainable: False
Jumlah parameter: 2422855


In [25]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

Epoch 1/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 446s 638ms/step - accuracy: 0.3332 - loss: 1.7009 - val_accuracy: 0.3878 - val_loss: 1.5894
Epoch 2/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 429s 619ms/step - accuracy: 0.3821 - loss: 1.5797 - val_accuracy: 0.3982 - val_loss: 1.5544
Epoch 3/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 502s 725ms/step - accuracy: 0.4093 - loss: 1.5283 - val_accuracy: 0.4122 - val_loss: 1.5290
Epoch 4/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 448s 645ms/step - accuracy: 0.4252 - loss: 1.4854 - val_accuracy: 0.4146 - val_loss: 1.5358
Epoch 5/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 469s 676ms/step - accuracy: 0.4421 - loss: 1.4440 - val_accuracy: 0.4141 - val_loss: 1.5394


In [30]:
from tensorflow.keras.layers import BatchNormalization

base_model.trainable = True

fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Batch Normalization tetap dibekukan
for layer in base_model.layers:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False

print("Total layer:", len(base_model.layers))

print(
    "Trainable layers:",
    sum(layer.trainable for layer in base_model.layers)
)

Total layer: 154
Trainable layers: 36


In [31]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [32]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3
)

Epoch 1/3
686/686 ━━━━━━━━━━━━━━━━━━━━ 456s 650ms/step - accuracy: 0.4636 - loss: 1.3872 - val_accuracy: 0.4334 - val_loss: 1.4938
Epoch 2/3
686/686 ━━━━━━━━━━━━━━━━━━━━ 460s 664ms/step - accuracy: 0.4936 - loss: 1.3201 - val_accuracy: 0.4500 - val_loss: 1.4738
Epoch 3/3
686/686 ━━━━━━━━━━━━━━━━━━━━ 402s 580ms/step - accuracy: 0.5122 - loss: 1.2693 - val_accuracy: 0.4635 - val_loss: 1.4406


In [33]:
y_true_mobilenet = []
y_pred_mobilenet = []

for images, labels in val_dataset:
    predictions = model.predict(images, verbose=0)

    predicted_labels = np.argmax(predictions, axis=1)

    y_true_mobilenet.extend(labels.numpy())
    y_pred_mobilenet.extend(predicted_labels)

y_true_mobilenet = np.array(y_true_mobilenet)
y_pred_mobilenet = np.array(y_pred_mobilenet)

print("Jumlah data validation:", len(y_true_mobilenet))

Jumlah data validation: 5487


In [34]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true_mobilenet,
        y_pred_mobilenet,
        target_names=class_names,
        zero_division=0
    )
)

              precision    recall  f1-score   support

       angry       0.40      0.26      0.32       767
     disgust       0.00      0.00      0.00        76
        fear       0.42      0.27      0.33       777
       happy       0.53      0.71      0.61      1417
     neutral       0.44      0.42      0.43       973
         sad       0.37      0.46      0.41       943
    surprise       0.56      0.54      0.55       534

    accuracy                           0.46      5487
   macro avg       0.39      0.38      0.38      5487
weighted avg       0.45      0.46      0.45      5487

